In [2]:
!pip install transformers torch torchvision pdf2image pillow

  Using cached pdf2image-1.17.0-py3-none-any.whl.metadata (6.2 kB)
Using cached pdf2image-1.17.0-py3-none-any.whl (11 kB)



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
import cv2
import numpy as np
from pdf2image import convert_from_path
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# paths
pdf_path = r"C:\Users\pardh\Downloads\PDP\24-25 Assignment 1\Please upload your assignment file (in .pdf format) (File responses)\22BCS001 - ABHIGYAN NIRANJAN IIIT Dharwad.pdf"
poppler_path = r"C:\Users\pardh\Downloads\PDP\Release-26.02.0-0\poppler-26.02.0\Library\bin"
output_file = r"C:\Users\pardh\Downloads\PDP\output_lines.txt"

# model
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

pages = convert_from_path(pdf_path, dpi=300, poppler_path=poppler_path)

all_text = ""

for page_no, page in enumerate(pages):
    print(f"Page {page_no+1}")

    img = np.array(page)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # threshold
    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY_INV)

    # horizontal dilation (line grouping)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 5))
    dilated = cv2.dilate(thresh, kernel, iterations=1)

    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if h > 15 and w > 50:
            boxes.append((x, y, w, h))

    boxes = sorted(boxes, key=lambda b: b[1])

    page_text = f"\n========== PAGE {page_no+1} ==========\n"

    for x, y, w, h in boxes:
        line = img[y:y+h, x:x+w]
        line_pil = Image.fromarray(line).convert("RGB")

        pixel_values = processor(images=line_pil, return_tensors="pt").pixel_values
        generated_ids = model.generate(pixel_values, max_new_tokens=120)

        text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        text = text.strip()

        if len(text) > 1:
            page_text += text + "\n"

    all_text += page_text

print(all_text)

with open(output_file, "w", encoding="utf-8") as f:
    f.write(all_text)

print("saved")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 478/478 [00:00<00:00, 6748.91it/s]
[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Page 1
Page 2
Page 3
Page 4
Page 5
Page 6
Page 7

========== PAGE 1 ==========
CS304 Artificial
Intelligence
Assignment 2
Name : Abhigyan Niranjan
Roll no : 22BCS001
Q1 : " As per the law , it is a crime for an American to sell weapons
A , an enemy of America ,
Country
has some
to hostile nations .
missiles , and all the missiles were sold to it by Robert , who is an
American citizen . "
Now , prove that " Robert is a criminal . "
Given :
weapon ( R )
hostile ( Q )
american CP )
criminal ( p ) .
r )
sells (p.
sp
9.
America )
enemy (A ,
t1 )
owns (A ,
missile (t. )
sells ( Robert ,
p )
owns (A ,
missile (p )
vp
sp
p )
a.
american ( Robert )
Additionally :
weapon ( p ) .
missileCp )
sp

========== PAGE 2 ==========
hostile (p )
America )
enemy Up .
sp
Proof using Forward Checking :
Step 1
0 0
Step 2
52 5
Step 3
2 1

========== PAGE 3 ==========
Q2 : Assume the following facts :
John likes all kinds of food .
Apple and vegetable are food .
Anything anyone eats and is not killed is food .
